In [1]:
# 1. Libraries

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import dash
from dash import dcc, html, Input, Output, State
import dash_ag_grid as dag
import dash_bootstrap_components as dbc
import base64
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import plotly.graph_objects as go

In [2]:
# 2. Import File

path_to_file = "https://raw.githubusercontent.com/cesarlarasantana/VDS_2526_G04_Football/refs/heads/main/Datasets"

df_country = pd.read_csv(path_to_file+"/Country.csv")
df_match_goals = pd.read_csv(path_to_file+"/Match_Goals.csv")
df_match_shots_on = pd.read_csv(path_to_file+"/Match_Shots_On.csv")
df_match_fouls = pd.read_csv(path_to_file+"/Match_Fouls_Committed.csv")
df_match_cards = pd.read_csv(path_to_file+"/Match_Cards.csv")
df_team = pd.read_csv(path_to_file+"/Team.csv")
df_match = pd.read_csv(path_to_file+"/Match.csv")
df_player = pd.read_csv(path_to_file+"/Player.csv")
df_player_att = pd.read_csv(path_to_file+"/Player_Attributes.csv", sep= ';')
df_position_ref = pd.read_csv(path_to_file+"/PositionReference.csv")

C:\Users\pci\AppData\Local\Temp\ipykernel_26424\3388209417.py:8: DtypeWarning: Columns (0: player1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_match_fouls = pd.read_csv(path_to_file+"/Match_Fouls_Committed.csv")


In [3]:
# 3. Data Preparation for Match Summary Dataset

#3.1 GOALS
### Only valid goals: 1) n: normal, 2) p: penalty

df_match_goals_valid =  df_match_goals[(df_match_goals['goal_type'] == "n") | (df_match_goals['goal_type'] == "p")].reset_index()
agg_df_match_goals = df_match_goals_valid.groupby(['match_id', 'team']).agg({'goal_type':'count'}).reset_index()

#3.2 FOULS
agg_df_match_fouls = df_match_fouls.groupby(['match_id', 'team']).agg({'elapsed':'count'}).reset_index()

#3.3 CARDS
agg_df_match_cards = df_match_cards.groupby(['match_id', 'team']).agg({'elapsed':'count'}).reset_index()

#3.4 SHOTS-ON

agg_df_match_shots_on = df_match_shots_on.groupby(['match_id', 'team']).agg({'elapsed':'count'}).reset_index()

#3.5 MATCH OUTCOME

df_match['winning_team_id'] = 9999999
df_match['winning_team_goals'] = 0
df_match['lossing_team_id'] = 9999999
df_match['lossing_team_goals'] = 0
df_match['draw_team1_id'] = 0
df_match['draw_team2_id'] = 0
df_match['draw_team1_goals'] = 0
df_match['draw_team2_goals'] = 0

for i in range(0, len(df_match)):
    if(df_match.iloc[i]['home_team_goal'] > df_match.iloc[i]['away_team_goal']):
        df_match.at[i, 'winning_team_id'] = df_match.iloc[i]['home_team_api_id']
        df_match.at[i, 'winning_team_goals'] = df_match.iloc[i]['home_team_goal']

        df_match.at[i, 'lossing_team_id'] = df_match.iloc[i]['away_team_api_id']
        df_match.at[i, 'lossing_team_goals'] = df_match.iloc[i]['away_team_goal']

    elif(df_match.iloc[i]['home_team_goal'] < df_match.iloc[i]['away_team_goal']):
        df_match.at[i, 'winning_team_id'] = df_match.iloc[i]['away_team_api_id']
        df_match.at[i, 'winning_team_goals'] = df_match.iloc[i]['away_team_goal']

        df_match.at[i, 'lossing_team_id'] = df_match.iloc[i]['home_team_api_id']
        df_match.at[i, 'lossing_team_goals'] = df_match.iloc[i]['home_team_goal']
    else:
        df_match.at[i, 'draw_team1_id'] = df_match.iloc[i]['home_team_api_id']
        df_match.at[i, 'draw_team2_id'] = df_match.iloc[i]['away_team_api_id']

        df_match.at[i, 'draw_team1_goals'] = df_match.iloc[i]['home_team_goal']
        df_match.at[i, 'draw_team2_goals'] = df_match.iloc[i]['away_team_goal']


In [4]:

# 3.6 Analysis Per Game 

## Winning Result per Game, Season and Team
agg_df_match_team_win = df_match.groupby(['match_api_id', 'winning_team_id', 'season']).agg({'stage':'count',
                                                                             'winning_team_goals': 'sum'}).reset_index()
agg_df_match_team_win = agg_df_match_team_win.rename({'winning_team_id' : 'team',
                                                      'stage': 'num',
                                                      'winning_team_goals': 'sum_goals'
                                                      }, axis=1)
agg_df_match_team_win['game_result'] = '1_Win'
agg_df_match_team_win = agg_df_match_team_win[agg_df_match_team_win['team'] != 9999999]

## Lossing Result per Game, Season and Team
agg_df_match_team_los = df_match.groupby(['match_api_id', 'lossing_team_id', 'season']).agg({'stage':'count',
                                                                             'lossing_team_goals': 'sum'}).reset_index()
agg_df_match_team_los = agg_df_match_team_los.rename({'lossing_team_id' : 'team',
                                                      'stage': 'num',
                                                      'lossing_team_goals': 'sum_goals'
                                                      }, axis=1)
agg_df_match_team_los['game_result'] = '2_Loss'
agg_df_match_team_los = agg_df_match_team_los[agg_df_match_team_los['team'] != 9999999]

## Draw Result Game, Season and Team - DRAW 1

agg_df_match_team_draw_1 = df_match.groupby(['match_api_id', 'draw_team1_id', 'season']).agg({'stage':'count',
                                                                             'draw_team1_goals': 'sum'}).reset_index()
agg_df_match_team_draw_1 = agg_df_match_team_draw_1.rename({'draw_team1_id' : 'team',
                                                      'stage': 'num',
                                                      'draw_team1_goals': 'sum_goals'
                                                      }, axis=1)
agg_df_match_team_draw_1['game_result'] = '3_Draw'
agg_df_match_team_draw_1 = agg_df_match_team_draw_1[agg_df_match_team_draw_1['team'] != 0]


## Draw Result Game, Season and Team - DRAW 2

agg_df_match_team_draw_2 = df_match.groupby(['match_api_id', 'draw_team2_id', 'season']).agg({'stage':'count',
                                                                             'draw_team2_goals': 'sum'}).reset_index()
agg_df_match_team_draw_2 = agg_df_match_team_draw_2.rename({'draw_team2_id' : 'team',
                                                      'stage': 'num',
                                                      'draw_team2_goals': 'sum_goals'
                                                      }, axis=1)
agg_df_match_team_draw_2['game_result'] = '3_Draw'
agg_df_match_team_draw_2 = agg_df_match_team_draw_2[agg_df_match_team_draw_2['team'] != 0]

agg_df_match_team = pd.concat([agg_df_match_team_win, agg_df_match_team_los, agg_df_match_team_draw_1, agg_df_match_team_draw_2]).reset_index()

agg_df_match_team = agg_df_match_team.sort_values(by = ['season','match_api_id', 'team']).reset_index()


In [5]:
# New Data Preparation
## For every Team, each season, count the number of victories, losses and draws

# Win Data
agg_df_match_team_w = agg_df_match_team[agg_df_match_team['game_result'] == '1_Win']
agg_team_wins = agg_df_match_team_w.groupby(['season', 'team']).agg({'sum_goals': 'sum',
                                                                     'num': 'count'}).reset_index()
agg_team_wins = agg_team_wins.rename({'sum_goals': 'sum_goals_win',
                                      'num' : 'num_games_wins'}, axis=1)

# Loss Data
agg_df_match_team_l = agg_df_match_team[agg_df_match_team['game_result'] == '2_Loss']
agg_team_losses = agg_df_match_team_l.groupby(['season', 'team']).agg({'sum_goals': 'sum',
                                                                     'num': 'count'}).reset_index()

agg_team_losses = agg_team_losses.rename({'sum_goals': 'sum_goals_losses',
                                         'num' : 'num_games_losses'}, axis=1)

# Draw Data
agg_df_match_team_d = agg_df_match_team[agg_df_match_team['game_result'] == '3_Draw']
agg_team_draws = agg_df_match_team_d.groupby(['season', 'team']).agg({'sum_goals': 'sum',
                                                                     'num': 'count'}).reset_index()

agg_team_draws = agg_team_draws.rename({'sum_goals': 'sum_goals_draws',
                                        'num' : 'num_games_draws'}, axis=1)

## Merging The three Data Frames

df_1 = agg_team_wins.merge(agg_team_losses, on=['season', 'team'], how = 'outer').reset_index()
df_1_team_summary = df_1.merge(agg_team_draws, on=['season', 'team'], how = 'outer').reset_index()
df_1_team_summary['total_games'] = df_1_team_summary['num_games_wins'] + df_1_team_summary['num_games_losses'] + df_1_team_summary['num_games_draws']
df_1_team_summary['prct_win'] = round((df_1_team_summary['num_games_wins']/df_1_team_summary['total_games'])*100, 2)

df_1_team_summary = df_1_team_summary.rename({'team': 'team_api_id'}, axis=1)

## Identifying which Tier does the Team belongs

df_1_team_summary['tier_group'] = ""

for i in range(0, len(df_1_team_summary)):
    if(df_1_team_summary.iloc[i]['prct_win'] > 90):
        df_1_team_summary.at[i, 'tier_group'] = "1_Top_10%"
    elif(df_1_team_summary.iloc[i]['prct_win'] > 80):
        df_1_team_summary.at[i, 'tier_group'] = "2_Top_20%"
    elif(df_1_team_summary.iloc[i]['prct_win'] > 70):
        df_1_team_summary.at[i, 'tier_group'] = "3_Top_30%"
    elif(df_1_team_summary.iloc[i]['prct_win'] > 60):
        df_1_team_summary.at[i, 'tier_group'] = "4_Top_40%"
    elif(df_1_team_summary.iloc[i]['prct_win'] > 50):
        df_1_team_summary.at[i, 'tier_group'] = "5_Top_50%"
    else:
        df_1_team_summary.at[i, 'tier_group'] = "6_Below_50%"

## Include the Team Name

df_1_team_summary = df_1_team_summary.merge(df_team[['team_api_id', 'team_long_name']], on='team_api_id', how='left')
df_1_team_summary = df_1_team_summary.drop({'index', 'level_0'}, axis=1)

In [6]:
## BVB Dortmund Players Performance in 2015/ 2016 Season

## 1. Merge Match Information (Which Players Played, Position in Field) with Players Name and AttributeError

def create_df_position_player(arg_plyr_num: str, position_ply_X: str, position_ply_Y: str, out_tb_name: str):

    out_tb_name = df_match.groupby(['season', 'home_team_api_id', arg_plyr_num]).agg({
    position_ply_X: 'max',
    position_ply_Y: 'max'}).reset_index()

    return pd.DataFrame(out_tb_name)

df_mgr_hp1 = create_df_position_player('home_player_1', 'home_player_X1', 'home_player_Y1', 'df_mrg_hp1')
df_mgr_hp2 = create_df_position_player('home_player_2', 'home_player_X2', 'home_player_Y2', 'df_mrg_hp2')
df_mgr_hp3 = create_df_position_player('home_player_3', 'home_player_X3', 'home_player_Y3', 'df_mrg_hp3')
df_mgr_hp4 = create_df_position_player('home_player_4', 'home_player_X4', 'home_player_Y4', 'df_mrg_hp4')
df_mgr_hp5 = create_df_position_player('home_player_5', 'home_player_X5', 'home_player_Y5', 'df_mrg_hp5')
df_mgr_hp6 = create_df_position_player('home_player_6', 'home_player_X6', 'home_player_Y6', 'df_mrg_hp6')
df_mgr_hp7 = create_df_position_player('home_player_7', 'home_player_X7', 'home_player_Y7', 'df_mrg_hp7')
df_mgr_hp8 = create_df_position_player('home_player_8', 'home_player_X8', 'home_player_Y8', 'df_mrg_hp8')
df_mgr_hp9 = create_df_position_player('home_player_9', 'home_player_X9', 'home_player_Y9', 'df_mrg_hp9')
df_mgr_hp10 = create_df_position_player('home_player_10', 'home_player_X10', 'home_player_Y10', 'df_mrg_hp10')
df_mgr_hp11 = create_df_position_player('home_player_11', 'home_player_X11', 'home_player_Y11', 'df_mrg_hp11')

## 2. Unify columns Name in order to generate one file DataFrame


In [7]:
## 3. Unify columns Name in order to generate one file DataFrame

df_mgr_hp1 = df_mgr_hp1.rename(columns ={'home_player_1': 'home_player', 'home_player_X1': 'player_pos_x', 'home_player_Y1': 'player_pos_y'})
df_mgr_hp2 = df_mgr_hp2.rename(columns ={'home_player_2': 'home_player', 'home_player_X2': 'player_pos_x', 'home_player_Y2': 'player_pos_y'})
df_mgr_hp3 = df_mgr_hp3.rename(columns ={'home_player_3': 'home_player', 'home_player_X3': 'player_pos_x', 'home_player_Y3': 'player_pos_y'})
df_mgr_hp4 = df_mgr_hp4.rename(columns ={'home_player_4': 'home_player', 'home_player_X4': 'player_pos_x', 'home_player_Y4': 'player_pos_y'})
df_mgr_hp5 = df_mgr_hp5.rename(columns ={'home_player_5': 'home_player', 'home_player_X5': 'player_pos_x', 'home_player_Y5': 'player_pos_y'})
df_mgr_hp6 = df_mgr_hp6.rename(columns ={'home_player_6': 'home_player', 'home_player_X6': 'player_pos_x', 'home_player_Y6': 'player_pos_y'})
df_mgr_hp7 = df_mgr_hp7.rename(columns ={'home_player_7': 'home_player', 'home_player_X7': 'player_pos_x', 'home_player_Y7': 'player_pos_y'})
df_mgr_hp8 = df_mgr_hp8.rename(columns ={'home_player_8': 'home_player', 'home_player_X8': 'player_pos_x', 'home_player_Y8': 'player_pos_y'})
df_mgr_hp9 = df_mgr_hp9.rename(columns ={'home_player_9': 'home_player', 'home_player_X9': 'player_pos_x', 'home_player_Y9': 'player_pos_y'})
df_mgr_hp10 = df_mgr_hp10.rename(columns ={'home_player_10': 'home_player', 'home_player_X10': 'player_pos_x', 'home_player_Y10': 'player_pos_y'})
df_mgr_hp11 = df_mgr_hp11.rename(columns ={'home_player_11': 'home_player', 'home_player_X11': 'player_pos_x', 'home_player_Y11': 'player_pos_y'})



In [8]:
## 4. Concatenate All Files and Bring the Position Category

df_mgr_all_pos = pd.concat([df_mgr_hp1, df_mgr_hp2, df_mgr_hp3, df_mgr_hp4, df_mgr_hp5, df_mgr_hp6, df_mgr_hp7,
                            df_mgr_hp8, df_mgr_hp9, df_mgr_hp10, df_mgr_hp11])

df_mgr_all_pos_cat = df_mgr_all_pos.merge(df_position_ref, on=['player_pos_x', 'player_pos_y'], how='left')

df_mgr_all_pos_cat['year_ranking'] = 0

for i in range (0, len(df_mgr_all_pos_cat)):
    df_mgr_all_pos_cat.at[i, 'year_ranking'] = int(df_mgr_all_pos_cat.iloc[i]['season'][5:9])

df_mgr_all_pos_cat['home_player'] = df_mgr_all_pos_cat['home_player'].astype(int)

df_mgr_all_pos_cat.rename(columns ={'home_player': 'player_api_id',
                                    'home_team_api_id': 'team_api_id'}, inplace= True)

## 5. Bring the Maximum Ranking - Performance By Player in the Year evaluated

df_player_att_adj = df_player_att

df_player_att_adj['date'] = pd.to_datetime(df_player_att_adj['date'], errors='coerce')

df_player_att_adj['year_ranking'] = 0
df_player_att_adj['year_ranking'] = df_player_att_adj['date'].dt.year

df_agg_player_att_adj_year = df_player_att_adj.groupby(['player_api_id', 'year_ranking']).agg({
    'overall_rating': 'max'}).reset_index()

# 6. Merge Position By Player By year with Overal Performance

df_mgr_all_pos_cat_merge = df_mgr_all_pos_cat.merge(df_agg_player_att_adj_year, on = ['player_api_id', 'year_ranking'], how = 'left')

# 7. Merge Team's Name. Simplify the Data Frame

df_team_smp = df_team[['team_api_id', 'team_long_name']]

df_mgr_all_pos_cat_merge = df_mgr_all_pos_cat_merge.merge(df_team_smp, on = ['team_api_id'], how = 'left')

# 8. Merge Player's name Name. Simplify the Data Frame

df_player_smp = df_player[['player_api_id', 'player_name']]

df_mgr_all_pos_cat_merge = df_mgr_all_pos_cat_merge.merge(df_player_smp, on = ['player_api_id'], how = 'left')


C:\Users\pci\AppData\Local\Temp\ipykernel_26424\1017421471.py:22: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_player_att_adj['date'] = pd.to_datetime(df_player_att_adj['date'], errors='coerce')


In [9]:
# 9. Create App

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

image_filename = 'C:/Cesar/MSC/Visualization_DS/Visualization_Project/BVB_Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

df_report_ply= df_mgr_all_pos_cat_merge[['team_long_name', 'player_name', 'role_xy', 'year_ranking', 'overall_rating']]
df_report_ply= df_report_ply[df_report_ply['year_ranking'] == 2016]
df_report_ply = df_report_ply.dropna()

app.layout = html.Div(
    children= [
    dbc.Row([
        html.H1("Borussia Dortmund (BVB) Strategic Game Analysis"),
        html.H2("Analysis Before The Game. End of 2016 Season"),
        html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
                 style={'height': '80px', 'width': '10%'})
    ]),
    
    # Dropdown for version selection
    html.Div([
        html.Label("Select Team:"),
            dcc.Dropdown(id='category-dropdown',
                         options=[{"label": name, "value": name} for name in df_report_ply["team_long_name"].unique()],
                         clearable=False),
            dbc.Button("Check Team Average Rating", id="update-btn", style={"backgroundColor": "#f3b576", "borderColor": "#f0ec22"}, className="mt-2"),

    ], style={'marginBottom': '20px'}),

    # AG Grid to display the filtered data
    dbc.Col(
        [
        html.H3("1. Previous Season Performance"),
        dbc.Card([dbc.CardBody([html.H4("Team Win Percentage in 2016"), html.H2(id="avg-card-team", style={'color': "#0227face"})])]),
        dbc.Card([dbc.CardBody([html.H4("Average Win Percentage - All teams"), html.H2(id="avg-card-all",  style={'color': "#0227face"})])]),
        
        html.H3("2. Team Roster Overal Rating"),
        dag.AgGrid(

            id='summary-grid',
            columnDefs=[
                {"field": "team_long_name", "headerName": "Team"},
                {"field": "player_name", "headerName": "Player"},
                {"field": "role_xy", "headerName": "Position"},
                {"field": "year_ranking", "headerName" : "Year"},
                {"field": "overall_rating", "headerName": "Average Rating"}
            ],
            dashGridOptions={"pagination": True},
            defaultColDef={"resizable": True, "sortable": True, "filter": True},
        ),
        html.H3("3. Average Rating by Position - Benchmark: All Players"),
        dcc.Graph(id='bar-chart')
            ]
        ),
    ]
    )

# Callback to update grid data based on selection

@app.callback(
    [Output("avg-card-team", "children"),
    Output("avg-card-all", "children"),
    Output('summary-grid', 'rowData'),
    Output('bar-chart', 'figure')],
    Input('update-btn', 'n_clicks'),
    State('category-dropdown', 'value'),
)

def update_rollback_view_ap3(n_clicks, selected_team):
    
    df_team_selected = df_report_ply[(df_report_ply['team_long_name'] == selected_team)]
    df_team_other = df_report_ply[(df_report_ply['team_long_name'] != selected_team)]

    agg_df_own_team = df_team_selected.groupby('role_xy').agg({'overall_rating' : 'mean'}).reset_index()
    agg_df_team_other = df_team_other.groupby('role_xy').agg({'overall_rating' : 'mean'}).reset_index()
    agg_df_team_other.rename(columns ={'overall_rating': 'general_rating'}, inplace= True)
    
    agg_df_own_team = agg_df_own_team.merge(agg_df_team_other, on='role_xy', how ='left').reset_index()

    # Figure 1 - Recent Year Average Rating for players

    fig_bar = px.bar(data_frame=agg_df_own_team, x='role_xy', y='overall_rating')
    fig_bar.update_traces(marker_color="#3143e2")

    fig_bar.add_trace(
        go.Scatter(
            x=agg_df_own_team['role_xy'], 
            y=agg_df_own_team['general_rating'], 
            mode='lines+markers', # Adds dots on the line for clarity
            name='Average Rate - All Teams - All Players'
        )
    )
    # Figure 2 - Last Season Productivity
    df_test_avg_team = df_1_team_summary[(df_1_team_summary['team_long_name'] == selected_team) &
                                    (df_1_team_summary['season'] == '2015/2016')]

    df_test_avg_all = df_1_team_summary[(df_1_team_summary['team_long_name'] != selected_team) &
                                    (df_1_team_summary['season'] == '2015/2016')]
    
    prct_win_2016_all = df_test_avg_all['prct_win'].mean()
    prct_win_2016_team = df_test_avg_team['prct_win'].mean()
    

    return f"{prct_win_2016_team:.1f}", f"{prct_win_2016_all:.1f}", df_team_selected.to_dict('records'), fig_bar

if __name__ == '__main__':
    app.run(port = 8075)